# 02 - CatBoost 5-Fold CV Baseline

Builds on `01_baseline.ipynb` (single 80/20 holdout, val AUC 0.941193).
That approach only gives one noisy point estimate and doesn't produce
out-of-fold predictions usable for later calibration/stacking. This
notebook replaces it with a proper 5-fold `StratifiedKFold`, using the
same raw features and the same CatBoost-native categorical handling.

No new feature engineering here on purpose — this notebook isolates the
effect of switching CV strategy. Feature engineering is the next step
(see the vault's Feature-Engineering-Playbook and Execution-Plan).

In [ ]:
import pandas as pd
import numpy as np
import time
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

print("Environment ready")

In [ ]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

TARGET = "Will_Buy_EV"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET].map({"No": 0, "Yes": 1})
X_test = test.drop(columns=["id"])

# NOTE: pandas 3 reads string columns as the new "str" dtype rather than
# "object" - select_dtypes(include=["object"]) alone silently misses them.
# Include "str" explicitly (and "category" in case dtypes are set upstream).
cat_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()

print("Train:", X.shape, " Test:", X_test.shape)
print("Categorical columns:", cat_cols)

Train: (668665, 13)  Test: (286571, 13)
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 5-fold CV training

Same hyperparameters as the Experiment 001 baseline (`depth=7`,
`learning_rate=0.05`), but `iterations` raised to 2000 with
`early_stopping_rounds=50` per fold rather than fixing it at 500 - the
right number of trees should come from early stopping, not be copied
from a different (smaller) training run. Each fold's model also predicts
on `X_test`; the 5 test-prediction columns are averaged at the end
instead of doing a separate full-data refit with a guessed iteration
count (the weakness flagged in Experiment 001).

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_preds = np.zeros((N_SPLITS, len(X_test)))
fold_aucs = []
fold_best_iters = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=7,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
    )
    model.fit(
        X_tr, y_tr,
        cat_features=cat_cols,
        eval_set=(X_va, y_va),
        early_stopping_rounds=50,
    )

    va_pred = model.predict_proba(X_va)[:, 1]
    oof_pred[va_idx] = va_pred

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_aucs.append(fold_auc)
    fold_best_iters.append(model.get_best_iteration())
    test_preds[fold] = model.predict_proba(X_test)[:, 1]

    print(f"Fold {fold}: AUC={fold_auc:.6f}  best_iter={model.get_best_iteration()}  elapsed={time.time()-t0:.0f}s")

print("Done in", round(time.time()-t0), "s")

Fold 0: AUC=0.940564  best_iter=1038  elapsed=137s
Fold 1: AUC=0.941328  best_iter=897  elapsed=255s
Fold 2: AUC=0.942533  best_iter=779  elapsed=357s
Fold 3: AUC=0.942292  best_iter=1060  elapsed=493s
Fold 4: AUC=0.941597  best_iter=761  elapsed=588s
Done in 588 s


In [ ]:
oof_auc = roc_auc_score(y, oof_pred)

print("Fold AUCs:", [round(a, 6) for a in fold_aucs])
print("Mean fold AUC:", np.mean(fold_aucs), " Std:", np.std(fold_aucs))
print("OOF AUC:", oof_auc)
print("Best iterations per fold:", fold_best_iters, " mean:", np.mean(fold_best_iters))

Fold AUCs: [0.940564, 0.941328, 0.942533, 0.942292, 0.941597]
Mean fold AUC: 0.941662727194531  Std: 0.0007039713449183326
OOF AUC: 0.941656249289744
Best iterations per fold: [1038, 897, 779, 1060, 761]  mean: 907.0


**Read on the numbers:** mean fold AUC (0.941663) and OOF AUC
(0.941656) agree to the 4th decimal - the CV is stable, not a fluke of
one lucky fold. Fold std (0.0007) is small relative to the score itself,
so this 0.9417 is a genuinely more trustworthy number than the single
holdout's 0.941193 was, even though the two are numerically close here.
Best iteration varies 761-1060 across folds (not fixed at a round
number like 500), which is exactly why early stopping per fold beats
copying an iteration count from a different run.

In [ ]:
test_pred_avg = test_preds.mean(axis=0)

submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_pred_avg
})

os.makedirs("../submissions", exist_ok=True)
submission.to_csv("../submissions/catboost_5fold_baseline.csv", index=False)

print("Submission shape:", submission.shape)
print("Missing values:", submission.isna().sum().sum())
submission.head()

Submission shape: (286571, 2)
Missing values: 0


       id  Will_Buy_EV\n0  668665     0.009813\n1  668666     0.021015\n2  668667     0.004438\n3  668668     0.002515\n4  668669     0.022735

## Experiment 002 - CatBoost 5-Fold CV

- Features: original features, ID removed (no engineering yet)
- CV: 5-fold `StratifiedKFold`, shuffle=True, random_state=42
- Model: CatBoostClassifier, depth=7, learning_rate=0.05, up to 2000
  iterations, early_stopping_rounds=50 per fold
- Test predictions: mean of the 5 fold models (no separate full-data refit)

**Mean fold AUC: 0.941663 (std 0.000704) - OOF AUC: 0.941656**

vs. Experiment 001 (single 80/20 holdout): 0.941193

The two scores are close, which is itself useful information: it
suggests Experiment 001's single split happened to be a fairly
representative one, not a lucky or unlucky draw. Going forward, treat
this notebook's OOF AUC as the reliable baseline number, and its
`oof_pred` / `test_preds` array as the inputs for isotonic calibration
and stacking later (see `04-Ensembling-Calibration` in the vault) - the
single-holdout model from `01_baseline.ipynb` can't produce those.

**Next step:** apply the feature-engineering backlog (digit
decomposition, frequency encoding, target encoding) on top of this same
CV structure, per Phase 2 of the execution plan.